# Plant Disease Detection with CNNs

Image classification of healthy and diseased plant leaves into 38 classes (14 crops) using the PlantVillage dataset.

Two convolutional neural networks are trained on the same data and compared:

1. **Custom CNN**: five convolutional blocks trained from scratch
2. **MobileNetV2**: a pretrained CNN fine-tuned with transfer learning

The model with the higher validation accuracy is exported to ONNX and used for plant disease detection on the Roots of Change website, where it runs directly in the browser.

**Run:** Kaggle or Google Colab with a GPU and internet access, then *Run all*. All figures, tables, the ONNX model and sample images are saved in `results/` and zipped as `plant_disease_results.zip`.

In [ ]:
!pip install -q onnx onnxruntime onnxscript

In [ ]:
import os, json, time, random, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 64
EPOCHS = 6
DATA_DIR = '/tmp/plantvillage'
OUT = 'results'
os.makedirs(f'{OUT}/samples', exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device, torch.cuda.get_device_name(0) if device == 'cuda' else '')
print('torch', torch.__version__, '| torchvision', torchvision.__version__)

## 1. Dataset

The PlantVillage dataset (54,305 colour images of healthy and diseased leaves) is used. If it is attached as a Kaggle input it is read from there; otherwise it is downloaded from its public Mendeley Data copy. Each class is split into training (80%), validation (10%) and test (10%) sets.

In [ ]:
import glob, zipfile, subprocess

DATA_URL = 'https://data.mendeley.com/public-files/datasets/tywbtsjrjv/files/d5652a28-c1d8-4b76-97f3-72fb80f94efc/file_downloaded'
RAW = '/tmp/raw'


def find_class_root(base):
    found = [d for d, subdirs, _ in os.walk(base) if sum('___' in s for s in subdirs) >= 30]
    preferred = [d for d in found if os.path.basename(d).lower() == 'color']
    return (preferred or found or [None])[0]


root = find_class_root('/kaggle/input') or find_class_root('/root/tensorflow_datasets/downloads') or find_class_root(RAW)
if root is None:
    archives = [p for p in glob.glob('/root/tensorflow_datasets/downloads/*') + glob.glob('/content/*.zip')
                if os.path.isfile(p) and os.path.getsize(p) > 5e8 and zipfile.is_zipfile(p)]
    archive = archives[0] if archives else '/tmp/plantvillage.zip'
    if not archives:
        subprocess.run(['wget', '-q', '-O', archive, DATA_URL], check=True)
    subprocess.run(['unzip', '-q', '-o', archive, '-d', RAW], check=True)
    root = find_class_root(RAW)

LABELS = sorted(s for s in os.listdir(root) if '___' in s)
rng = np.random.default_rng(SEED)
split_counts = {'train': 0, 'val': 0, 'test': 0}
class_counts = {}
for cls in LABELS:
    files = sorted(f for f in os.listdir(f'{root}/{cls}') if f.lower().endswith(('.jpg', '.jpeg', '.png')))
    class_counts[cls] = len(files)
    idx = rng.permutation(len(files))
    n = max(1, round(0.1 * len(files)))
    parts = {'test': idx[:n], 'val': idx[n:2 * n], 'train': idx[2 * n:]}
    for split, ids in parts.items():
        folder = f'{DATA_DIR}/{split}/{cls}'
        os.makedirs(folder, exist_ok=True)
        for k in ids:
            shutil.copy(f'{root}/{cls}/{files[k]}', f'{folder}/{files[k]}')
        split_counts[split] += len(ids)

counts = pd.Series(class_counts).sort_values()
print(len(LABELS), 'classes |', sum(split_counts.values()), 'images |', split_counts)

In [ ]:
plt.figure(figsize=(8, 11))
counts.plot.barh(color='seagreen')
plt.title('Images per class')
plt.xlabel('Number of images')
plt.tight_layout()
plt.savefig(f'{OUT}/class_distribution.png', dpi=200)
plt.show()

fig, axes = plt.subplots(3, 6, figsize=(16, 8.5))
train_root = f'{DATA_DIR}/train'
for ax, cls in zip(axes.ravel(), rng.choice(LABELS, 18, replace=False)):
    first = sorted(os.listdir(f'{train_root}/{cls}'))[0]
    ax.imshow(Image.open(f'{train_root}/{cls}/{first}'))
    ax.set_title(cls.replace('___', '\n').replace('_', ' '), fontsize=8)
    ax.axis('off')
plt.tight_layout()
plt.savefig(f'{OUT}/sample_images.png', dpi=150)
plt.show()

## 2. Data loaders and augmentation

Training images are augmented with random crops, flips, rotations and colour changes. Validation and test images are only resized to 224 × 224, which is the same preprocessing the website applies to uploaded photos.

In [ ]:
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_ds = datasets.ImageFolder(f'{DATA_DIR}/train', train_tf)
val_ds = datasets.ImageFolder(f'{DATA_DIR}/val', eval_tf)
test_ds = datasets.ImageFolder(f'{DATA_DIR}/test', eval_tf)
CLASSES = train_ds.classes

workers = min(8, os.cpu_count() or 2)
train_dl = DataLoader(train_ds, BATCH_SIZE, shuffle=True, num_workers=workers, pin_memory=True, persistent_workers=True)
val_dl = DataLoader(val_ds, 128, num_workers=workers, pin_memory=True, persistent_workers=True)
test_dl = DataLoader(test_ds, 128, num_workers=workers, pin_memory=True)
print(f'train {len(train_ds)} | validation {len(val_ds)} | test {len(test_ds)} | classes {len(CLASSES)}')

## 3. Models

* **Custom CNN:** five blocks of two 3 × 3 convolutions with batch normalization and ReLU, each followed by max pooling, then global average pooling, dropout and a dense output layer.
* **MobileNetV2:** pretrained on ImageNet; the final layer is replaced with a 38-class output and the whole network is fine-tuned.

In [ ]:
def conv_block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1, bias=False), nn.BatchNorm2d(c_out), nn.ReLU(inplace=True),
        nn.MaxPool2d(2),
    )


class PlantCNN(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(conv_block(3, 32), conv_block(32, 64), conv_block(64, 128),
                                      conv_block(128, 256), conv_block(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.classifier(self.pool(self.features(x)))


def build_mobilenet(n_classes):
    net = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.IMAGENET1K_V1)
    net.classifier[1] = nn.Linear(net.last_channel, n_classes)
    return net


nets = {'Custom CNN': PlantCNN(len(CLASSES)).to(device),
        'MobileNetV2': build_mobilenet(len(CLASSES)).to(device)}
for name, net in nets.items():
    print(f'{name}: {sum(p.numel() for p in net.parameters()) / 1e6:.2f} M parameters')

## 4. Training

Both models are trained on exactly the same batches with AdamW, a one-cycle learning-rate schedule, label smoothing and mixed precision. The weights with the best validation accuracy are kept for each model.

In [ ]:
def make_optimizer(name, net):
    if name == 'MobileNetV2':
        groups = [{'params': net.features.parameters(), 'lr': 3e-4},
                  {'params': net.classifier.parameters(), 'lr': 1e-3}]
        max_lr = [3e-4, 1e-3]
    else:
        groups = [{'params': net.parameters(), 'lr': 2e-3}]
        max_lr = [2e-3]
    opt = torch.optim.AdamW(groups, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=max_lr, epochs=EPOCHS, steps_per_epoch=len(train_dl))
    return {'opt': opt, 'sched': sched, 'scaler': torch.amp.GradScaler('cuda', enabled=device == 'cuda')}


criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
state = {name: make_optimizer(name, net) for name, net in nets.items()}
use_amp = device == 'cuda'


def train_epoch():
    for net in nets.values():
        net.train()
    stats = {name: [0.0, 0, 0] for name in nets}
    for x, y in train_dl:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        for name, net in nets.items():
            s = state[name]
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                out = net(x)
                loss = criterion(out, y)
            s['opt'].zero_grad(set_to_none=True)
            s['scaler'].scale(loss).backward()
            s['scaler'].step(s['opt'])
            s['scaler'].update()
            s['sched'].step()
            stats[name][0] += loss.item() * y.size(0)
            stats[name][1] += (out.argmax(1) == y).sum().item()
            stats[name][2] += y.size(0)
    return {n: (v[0] / v[2], v[1] / v[2]) for n, v in stats.items()}


@torch.no_grad()
def evaluate(loader):
    for net in nets.values():
        net.eval()
    stats = {name: [0.0, 0, 0] for name in nets}
    for x, y in loader:
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        for name, net in nets.items():
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                out = net(x)
                loss = criterion(out, y)
            stats[name][0] += loss.item() * y.size(0)
            stats[name][1] += (out.argmax(1) == y).sum().item()
            stats[name][2] += y.size(0)
    return {n: (v[0] / v[2], v[1] / v[2]) for n, v in stats.items()}


history = []
best = {name: (0.0, None) for name in nets}
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr = train_epoch()
    va = evaluate(val_dl)
    for name, net in nets.items():
        history.append({'model': name, 'epoch': epoch, 'train_loss': tr[name][0], 'train_acc': tr[name][1],
                        'val_loss': va[name][0], 'val_acc': va[name][1]})
        if va[name][1] > best[name][0]:
            best[name] = (va[name][1], {k: v.detach().cpu().clone() for k, v in net.state_dict().items()})
    print(f'epoch {epoch}/{EPOCHS} ({time.time() - t0:.0f} s)  ' +
          '   '.join(f'{n}: train acc {tr[n][1]:.4f}, val acc {va[n][1]:.4f}' for n in nets))

for name, net in nets.items():
    net.load_state_dict(best[name][1])
history = pd.DataFrame(history)
history.to_csv(f'{OUT}/training_history.csv', index=False)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for name, h in history.groupby('model'):
    axes[0].plot(h.epoch, h.train_acc, '--', marker='o', label=f'{name} (train)')
    axes[0].plot(h.epoch, h.val_acc, '-', marker='o', label=f'{name} (validation)')
    axes[1].plot(h.epoch, h.train_loss, '--', marker='o', label=f'{name} (train)')
    axes[1].plot(h.epoch, h.val_loss, '-', marker='o', label=f'{name} (validation)')
axes[0].set_title('Accuracy')
axes[1].set_title('Loss')
for ax in axes:
    ax.set_xlabel('Epoch')
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(f'{OUT}/training_curves.png', dpi=200)
plt.show()

## 5. Test results and model comparison

In [ ]:
@torch.no_grad()
def predict(net, loader):
    net.eval()
    preds, labels = [], []
    for x, y in loader:
        preds.append(net(x.to(device)).argmax(1).cpu().numpy())
        labels.append(y.numpy())
    return np.concatenate(preds), np.concatenate(labels)


rows, test_preds = [], {}
for name, net in nets.items():
    p, y_true = predict(net, test_dl)
    test_preds[name] = p
    rows.append({'Model': name,
                 'Parameters (M)': round(sum(q.numel() for q in net.parameters()) / 1e6, 2),
                 'Best validation accuracy': round(best[name][0], 4),
                 'Test accuracy': round(accuracy_score(y_true, p), 4),
                 'Test macro F1': round(f1_score(y_true, p, average='macro'), 4)})
comparison = pd.DataFrame(rows).set_index('Model')
comparison.to_csv(f'{OUT}/model_comparison.csv')
print(comparison.to_string())

SELECTED = max(nets, key=lambda n: best[n][0])
print('\nModel used on the website (highest validation accuracy):', SELECTED)

ax = comparison[['Test accuracy', 'Test macro F1']].plot.bar(figsize=(7, 4.5), rot=0, color=['seagreen', 'steelblue'])
ax.set_ylim(max(0.0, comparison[['Test accuracy', 'Test macro F1']].values.min() - 0.05), 1.0)
for c in ax.containers:
    ax.bar_label(c, fmt='%.4f', fontsize=8)
ax.set_title('Test set comparison')
plt.tight_layout()
plt.savefig(f'{OUT}/model_comparison.png', dpi=200)
plt.show()

In [ ]:
y_pred = test_preds[SELECTED]
report = classification_report(y_true, y_pred, target_names=CLASSES, digits=4)
with open(f'{OUT}/classification_report.txt', 'w') as f:
    f.write(f'{SELECTED} (test set)\n\n{report}')
print(report)

cm = confusion_matrix(y_true, y_pred, normalize='true')
short = [c.replace('___', ': ').replace('_', ' ') for c in CLASSES]
plt.figure(figsize=(16, 14))
sns.heatmap(cm, cmap='Greens', xticklabels=short, yticklabels=short, vmin=0, vmax=1, cbar_kws={'shrink': 0.6})
plt.title(f'Normalized confusion matrix: {SELECTED} (test set)')
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.xticks(fontsize=7)
plt.yticks(fontsize=7)
plt.tight_layout()
plt.savefig(f'{OUT}/confusion_matrix.png', dpi=200)
plt.show()

## 6. Export to ONNX for the website

The selected model is exported to ONNX and checked against the PyTorch model with ONNX Runtime.

In [ ]:
import onnxruntime as ort

export_net = nets[SELECTED].to('cpu').eval()
dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
onnx_path = f'{OUT}/plant_disease_model.onnx'
try:
    torch.onnx.export(export_net, dummy, onnx_path, input_names=['input'], output_names=['logits'],
                      opset_version=17, dynamo=False)
except TypeError:
    torch.onnx.export(export_net, dummy, onnx_path, input_names=['input'], output_names=['logits'], opset_version=17)
except Exception as e:
    print('Using the dynamo exporter:', e)
    torch.onnx.export(export_net, (dummy,), onnx_path, input_names=['input'], output_names=['logits'],
                      dynamo=True, external_data=False)

sess = ort.InferenceSession(onnx_path, providers=['CPUExecutionProvider'])
input_name = sess.get_inputs()[0].name
x = torch.stack([test_ds[i][0] for i in np.linspace(0, len(test_ds) - 1, 20).astype(int)])
with torch.no_grad():
    ref = export_net(x).numpy()
got = np.concatenate([sess.run(None, {input_name: x[i:i + 1].numpy()})[0] for i in range(len(x))])
print('max difference PyTorch vs ONNX:', float(np.abs(ref - got).max()))
print('same predictions:', bool((ref.argmax(1) == got.argmax(1)).all()))
print(f'ONNX model size: {os.path.getsize(onnx_path) / 1e6:.1f} MB')

with open(f'{OUT}/classes.json', 'w') as f:
    json.dump(CLASSES, f, indent=1)

In [ ]:
picks = [('tomato', 'late'), ('potato', 'early'), ('apple', 'black'), ('corn', 'rust'), ('grape', 'black'), ('tomato', 'healthy')]
samples = []
for crop, cond in picks:
    cls = next((c for c in CLASSES if crop in c.lower() and cond in c.lower()), None)
    if cls is None:
        continue
    folder = f'{DATA_DIR}/test/{cls}'
    first = sorted(os.listdir(folder))[0]
    fname = ''.join(ch for ch in cls.lower().replace('___', '_').replace(' ', '_') if ch.isalnum() or ch == '_') + '.jpg'
    Image.open(f'{folder}/{first}').convert('RGB').save(f'{OUT}/samples/{fname}', quality=92)
    samples.append({'file': fname, 'label': cls})
with open(f'{OUT}/samples/samples.json', 'w') as f:
    json.dump(samples, f, indent=1)
print(samples)

## 7. Summary

In [ ]:
summary = {
    'dataset': 'PlantVillage, colour images (38 classes)',
    'classes': len(CLASSES),
    'images': split_counts,
    'image_size': IMG_SIZE,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'comparison': comparison.reset_index().to_dict(orient='records'),
    'selected_model': SELECTED,
    'onnx_file': 'plant_disease_model.onnx',
    'onnx_size_mb': round(os.path.getsize(onnx_path) / 1e6, 2),
    'preprocessing': {'resize': [IMG_SIZE, IMG_SIZE], 'scale': '0-1', 'mean': MEAN, 'std': STD, 'layout': 'NCHW RGB'},
    'versions': {'torch': torch.__version__, 'torchvision': torchvision.__version__, 'onnxruntime': ort.__version__},
}
with open(f'{OUT}/summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(json.dumps(summary, indent=2))

shutil.make_archive('plant_disease_results', 'zip', OUT)
print(sorted(os.listdir(OUT)))
try:
    from google.colab import files
    files.download('plant_disease_results.zip')
except ImportError:
    print('plant_disease_results.zip saved')